In [1]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser  # ✅ Works with your version

# 🔐 Load API keys
import os
from dotenv import load_dotenv
load_dotenv(dotenv_path=".env")
openai_api_key = "Open_api_key"
prompt = PromptTemplate.from_template("Translate to French: {text}")
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
parser = StrOutputParser()

chain = prompt | llm | parser

response = chain.invoke({"text": "Good morning"})
print(response)


Bonjour


In [2]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

prompt = PromptTemplate.from_template("List 5 programming languages, comma-separated.")
parser = CommaSeparatedListOutputParser()
chain = prompt | llm | parser

response = chain.invoke({})
print(response)  # ['Python', 'Java', 'C++', 'JavaScript', 'Ruby']


['Python', 'Java', 'C++', 'JavaScript', 'Ruby']


In [3]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

class ProductInfo(BaseModel):
    name: str = Field(description="Name of the product")
    price: float = Field(description="Price in INR")

parser = PydanticOutputParser(pydantic_object=ProductInfo)

prompt = PromptTemplate(
    template="Extract product name and price from: {text}\n{format_instructions}",
    input_variables=["text"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

chain = prompt | llm | parser

response = chain.invoke({
    "text": "The Redmi Note 12 is available for ₹14,999."
})
print(response)


name='Redmi Note 12' price=14999.0


In [5]:
# Run this ONCE
%pip install -q langchain-core pydantic

# Then this is your FULL new code - No ResponseSchema needed
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field

class CompanyInfo(BaseModel):
    company: str = Field(description="Name of the company")
    founder: str = Field(description="Name of the founder")

parser = JsonOutputParser(pydantic_object=CompanyInfo)

# This gives you same output as before
print(parser.get_format_instructions())

# Test it
test_json = '{"company": "Microsoft", "founder": "Bill Gates"}'
print(parser.parse(test_json))

Note: you may need to restart the kernel to use updated packages.
STRICT OUTPUT FORMAT:
- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.
- Do not wrap the JSON in Markdown or code fences (no ``` or ```json).
- Do not prepend or append any text (e.g., do not write "Here is the JSON:").
- The response must be a single top-level JSON value exactly as required by the schema (object/array/etc.), with no trailing commas or comments.

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]} the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema (shown in a code block for readability